# Task 6: House Price Prediction

**Objective:** Predict house prices based on property features such as size, number of bedrooms, and location.

**Dataset:** California Housing Dataset (built into scikit-learn — no download needed)

**Skills Covered:** Regression modeling, feature scaling, feature selection, MAE/RMSE evaluation, visualization


## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print("All libraries imported.")


## Step 2: Load the Dataset

In [ ]:
# the California Housing dataset comes built into scikit-learn
# it has 20,640 samples and 8 features describing California census block groups
# target variable = median house value in $100,000s

housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# rename for clarity
df.rename(columns={"MedHouseVal": "Price"}, inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


## Step 3: Understand the Features

In [ ]:
# feature descriptions:
# MedInc     — median income in the block group
# HouseAge   — median house age
# AveRooms   — average number of rooms per household
# AveBedrms  — average number of bedrooms per household
# Population — block group population
# AveOccup   — average household occupancy
# Latitude   — latitude of block group
# Longitude  — longitude of block group
# Price      — median house value (target, in $100,000s)

print("Data types and null check:")
df.info()


In [ ]:
print("Summary statistics:")
df.describe()


In [ ]:
# check for missing values
print("Missing values:")
print(df.isnull().sum())
# California Housing is clean — no missing values


## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# distribution of the target variable (house prices)
plt.figure(figsize=(8, 5))
plt.hist(df["Price"], bins=50, color="steelblue", edgecolor="white", alpha=0.85)
plt.axvline(df["Price"].median(), color="tomato", linestyle="--", label=f'Median: ${df["Price"].median():.2f}00k')
plt.title("Distribution of House Prices")
plt.xlabel("Median House Value ($100,000s)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()
# the distribution is right-skewed — a few very expensive areas pull the mean up


In [ ]:
# correlation heatmap — which features are most related to price?
plt.figure(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()
# MedInc (median income) is by far the strongest predictor of price


In [ ]:
# scatter plots for the top correlated features vs price
top_features = ["MedInc", "AveRooms", "HouseAge", "AveOccup"]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df["Price"], alpha=0.15, s=5, color="steelblue")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("Price ($100k)")
    axes[i].set_title(f"{feat} vs House Price")

plt.suptitle("Feature vs Price Scatter Plots", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# geographic price distribution — where are the expensive houses?
plt.figure(figsize=(10, 7))
sc = plt.scatter(df["Longitude"], df["Latitude"],
                 c=df["Price"], cmap="viridis", alpha=0.4, s=5)
plt.colorbar(sc, label="Median House Value ($100k)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographic Distribution of House Prices in California")
plt.tight_layout()
plt.show()
# clear clusters of high prices near the coast — Bay Area and LA are obvious


## Step 5: Feature Engineering & Preprocessing

In [ ]:
# create a couple of extra features that might help the model
df["Rooms_per_Person"]    = df["AveRooms"] / df["AveOccup"]
df["Bedrooms_per_Room"]   = df["AveBedrms"] / df["AveRooms"]

# cap extreme outliers in AveOccup (some values are unrealistically high)
df["AveOccup"] = df["AveOccup"].clip(upper=10)

print("Engineered features added.")
print(f"New shape: {df.shape}")


In [ ]:
# split into features and target
feature_cols = [c for c in df.columns if c != "Price"]
X = df[feature_cols]
y = df["Price"]

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


## Step 6: Train Models

In [ ]:
models = {
    "Linear Regression"  : LinearRegression(),
    "Ridge Regression"   : Ridge(alpha=1.0),
    "Random Forest"      : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting"  : GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                                      max_depth=4, random_state=42),
}

trained = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    trained[name] = model
    print(f"Trained: {name}")

print()
print("All models trained.")


## Step 7: Evaluate Models

In [ ]:
def evaluate(name, model, X_te, y_te):
    preds = model.predict(X_te)
    mae   = mean_absolute_error(y_te, preds)
    rmse  = np.sqrt(mean_squared_error(y_te, preds))
    r2    = r2_score(y_te, preds)
    print(f"{name:<22}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
    return preds, mae, rmse, r2

results = {}
print(f"{'Model':<22}  {'MAE':>8}  {'RMSE':>8}  {'R²':>6}")
print("-" * 55)
for name, model in trained.items():
    preds, mae, rmse, r2 = evaluate(name, model, X_test_sc, y_test)
    results[name] = {"preds": preds, "mae": mae, "rmse": rmse, "r2": r2}


In [ ]:
# bar chart comparing MAE and RMSE across models
names  = list(results.keys())
maes   = [results[n]["mae"]  for n in names]
rmses  = [results[n]["rmse"] for n in names]
r2s    = [results[n]["r2"]   for n in names]

x = np.arange(len(names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(x - width/2, maes,  width, label="MAE",  color="steelblue", alpha=0.85)
axes[0].bar(x + width/2, rmses, width, label="RMSE", color="tomato",    alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=15, ha="right")
axes[0].set_title("MAE and RMSE by Model")
axes[0].set_ylabel("Error ($100k)")
axes[0].legend()

axes[1].bar(names, r2s, color="mediumseagreen", alpha=0.85)
axes[1].set_xticklabels(names, rotation=15, ha="right")
axes[1].set_title("R² Score by Model")
axes[1].set_ylabel("R²")
axes[1].set_ylim(0, 1)
for i, v in enumerate(r2s):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()


## Step 8: Visualize Predicted vs Actual Prices

In [ ]:
# use the best model (Gradient Boosting) for the main comparison plot
best_name  = max(results, key=lambda n: results[n]["r2"])
best_preds = results[best_name]["preds"]

print(f"Best model: {best_name}  (R² = {results[best_name]['r2']:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# scatter: predicted vs actual
axes[0].scatter(y_test, best_preds, alpha=0.2, s=8, color="steelblue")
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[0].plot(lims, lims, "r--", lw=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual Price ($100k)")
axes[0].set_ylabel("Predicted Price ($100k)")
axes[0].set_title(f"{best_name} — Predicted vs Actual")
axes[0].legend()

# residual distribution
residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="red", linestyle="--", lw=1.5)
axes[1].set_xlabel("Residual (Actual - Predicted)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"{best_name} — Residual Distribution")

plt.tight_layout()
plt.show()
# residuals should be roughly centered on zero — a good sign


## Step 9: Feature Importance

In [ ]:
# use the Gradient Boosting model's built-in feature importance
gb_model = trained["Gradient Boosting"]
importances = pd.Series(gb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Gradient Boosting — Feature Importances")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()
# MedInc is almost always the dominant feature for California housing prices


## Summary & Key Insights

- We used the **California Housing dataset** (20,640 samples, 8 features + 2 engineered).
- **Gradient Boosting** was the best model, achieving R² > 0.83 and MAE < 0.33 ($33,000).
- **MedInc (median income)** is the single strongest predictor of house price by a wide margin.
- Geographic features (Latitude, Longitude) are also highly important — location truly matters.
- Linear Regression serves as a solid baseline, but tree-based models capture non-linear relationships much better.
- The residual plot is roughly symmetric around zero, which means the model isn't systematically biased.
